#### Steps taken:
1. Read silver races table.
2. Read silver circuits table.
3. Join data from both table using circuit_id.
4. select the required columns
  - races.season
  - races.round
  - races.race_name
  - races.race_date
  - circuits.circuit_name
  - circuits.locality
  - circuits.country
5. Write the transformed data to gold dim_races table.

In [0]:
%run ../environment_config

In [0]:
from pyspark.sql import functions as F

In [0]:
gold_table = f"{catalog}.{gold_schema}.dim_races"

In [0]:
circuits_df = spark.table(f"{catalog}.{silver_schema}.circuits")
races_df = spark.table(f"{catalog}.{silver_schema}.races")

In [0]:
circuits_df = circuits_df.withColumn("circuit_id", F.initcap("circuit_id"))

In [0]:
selected_cols = [
                   races_df.season, races_df.round, races_df.race_name, races_df.race_date,
                   circuits_df.circuit_name, circuits_df.locality, circuits_df.country
                ]
races_s_df = (
            races_df.join(
                        circuits_df,
                        circuits_df.circuit_id == races_df.circuit_id,
                        "inner"
                        )
                    .select(*selected_cols)
             )

In [0]:
display(races_s_df)

In [0]:
(
    races_s_df.write
              .format("delta")
              .mode("overwrite")
              .saveAsTable(gold_table)
)

In [0]:
display(spark.table(gold_table))